# 01 · Download and preprocess AERONET AOD Level 2.0 (Version 3)

**What this notebook does**

1. **Download** the NASA AERONET bulk archive (`.tar.gz`) into `DATA/RAW/` if it is not already there, extract it, and delete the archive.
2. **Read** every station file (`*.lev20`, one per AERONET site) and **merge** them into a single `pandas.DataFrame`.
3. **Export** one agent-ready Parquet table to `DATA/PROCESSED/`.

**Dataset**: AERONET **AOD Level 2.0** (cloud-screened *and* quality-assured), **Version 3**, **Daily averages**, all sites —
`https://aeronet.gsfc.nasa.gov/data_push/V3/AOD/AOD_Level20_Daily_V3.tar.gz` (~110 MB archive → ~570 MB extracted, ~1,485 sites, one row per site × day).

In [26]:
from pathlib import Path
import re
import tarfile
import time

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

In [27]:
ARCHIVE = "AOD_Level20_Daily_V3.tar.gz"
URL = "https://aeronet.gsfc.nasa.gov/data_push/V3/AOD/" + ARCHIVE

In [28]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [29]:
RAW_DIR       = PROJECT_ROOT / "DATA" / "RAW"
PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [30]:
TARBALL     = RAW_DIR / ARCHIVE                                  # deleted after extraction
EXTRACT_DIR = RAW_DIR / ARCHIVE.replace(".tar.gz", "")           # station files land here
OUT_PARQUET = PROCESSED_DIR / "AERONET_AOD_L2_Daily_V3.parquet"

In [31]:
print(f"URL          : {URL}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"EXTRACT_DIR  : {EXTRACT_DIR}")
print(f"OUT_PARQUET  : {OUT_PARQUET}")

URL          : https://aeronet.gsfc.nasa.gov/data_push/V3/AOD/AOD_Level20_Daily_V3.tar.gz
PROJECT_ROOT : /home/ubuntu/ai_for_aeronet
EXTRACT_DIR  : /home/ubuntu/ai_for_aeronet/DATA/RAW/AOD_Level20_Daily_V3
OUT_PARQUET  : /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_AOD_L2_Daily_V3.parquet


## 1 · Download (if needed) → extract → delete the archive

In [32]:
def download(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None
        with open(tmp, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for part in r.iter_content(chunk_size):
                fh.write(part)
                bar.update(len(part))
    tmp.replace(dest)


def extract_tar_gz(archive: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        try:
            tf.extractall(dest, filter="data")      
        except TypeError:                    
            tf.extractall(dest)


station_files = sorted(EXTRACT_DIR.rglob("*.lev20")) if EXTRACT_DIR.exists() else []

if station_files:
    print(f"Already extracted: {len(station_files)} station files in {EXTRACT_DIR}  →  skipping download.")
else:
    if TARBALL.exists():
        print(f"Found {TARBALL.name} ({TARBALL.stat().st_size / 1e6:,.0f} MB)  →  skipping download.")
    else:
        print(f"Downloading {URL}")
        t0 = time.time()
        download(URL, TARBALL)
        print(f"  done in {time.time() - t0:,.0f} s  ({TARBALL.stat().st_size / 1e6:,.0f} MB)")

    print(f"Extracting → {EXTRACT_DIR}")
    extract_tar_gz(TARBALL, EXTRACT_DIR)
    TARBALL.unlink()
    print(f"Deleted {TARBALL.name}")
    station_files = sorted(EXTRACT_DIR.rglob("*.lev20"))

print(f"{len(station_files):,} station files ready")

Already extracted: 1485 station files in /home/ubuntu/ai_for_aeronet/DATA/RAW/AOD_Level20_Daily_V3  →  skipping download.
1,485 station files ready


## 2 · Read every station file and merge

Each `*.lev20` file is one AERONET site:

```
line 1  AERONET Version 3
line 2  <site name>
line 3  Version 3: AOD Level 2.0
line 4  (description)
line 5  Contact: PI=<name>; PI Email=<email>
line 6  Daily Averages,UNITS can be found at,,, https://aeronet.gsfc.nasa.gov/new_web/units.html
line 7  <CSV header>
line 8+ <CSV rows>, missing values are -999
```

In [33]:
MISSING = -999.0

def read_station(path: Path) -> pd.DataFrame:
    # Parse one AERONET .lev20 file -> DataFrame (raw AERONET column names, -999 -> NaN,
    # plus `site`, `PI`, `PI_Email` taken from the file header).
    with open(path, encoding="latin-1") as fh:
        header = [next(fh).rstrip("\n") for _ in range(6)]
    site = header[1].strip()
    m = re.search(r"PI=(.*?);\s*PI Email=(.*)", header[4])
    pi, pi_email = (m.group(1).strip(), m.group(2).strip()) if m else (None, None)

    df = pd.read_csv(path, skiprows=6, encoding="latin-1", low_memory=False)

    # -999 → NaN in every numeric column (count columns never contain -999, so they stay integer)
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num] == MISSING)

    df.insert(0, "site", site)
    df["PI"] = pi
    df["PI_Email"] = pi_email
    return df


frames, failures = [], []
t0 = time.time()
for p in tqdm(station_files, desc="Reading stations"):
    try:
        frames.append(read_station(p))
    except Exception as exc:            
        failures.append((p.name, repr(exc)))

aeronet_raw = pd.concat(frames, ignore_index=True)
del frames

print(f"Merged {len(station_files) - len(failures):,} stations → {aeronet_raw.shape[0]:,} rows × {aeronet_raw.shape[1]} columns "
      f"in {time.time() - t0:,.0f} s")
if failures:
    print(f"\n{len(failures)} file(s) failed to parse:")
    for name, err in failures:
        print(f"  {name}: {err}")

Reading stations: 100%|██████████| 1485/1485 [00:39<00:00, 37.67it/s]


Merged 1,485 stations → 1,419,960 rows × 84 columns in 42 s


In [34]:
aeronet_raw

,site,Date(dd:mm:yyyy),Time(hh:mm:ss),Day_of_Year,AOD_1640nm,AOD_1020nm,AOD_870nm,AOD_865nm,AOD_779nm,AOD_675nm,...,N[340-440_Angstrom_Exponent],N[440-675_Angstrom_Exponent[Polar]],Data_Quality_Level,AERONET_Instrument_Number,AERONET_Site_Name,Site_Latitude(Degrees),Site_Longitude(Degrees),Site_Elevation(m),PI,PI_Email
0,AAOT,04:10:2017,12:00:00,277,NaN,0.169420,NaN,0.227937,0.270857,NaN,...,0,0,lev20,1001,AAOT,45.3139,12.5083,10.0,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,11:10:2017,12:00:00,284,NaN,0.121093,NaN,0.165021,0.201501,NaN,...,0,0,lev20,1001,AAOT,45.3139,12.5083,10.0,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,12:10:2017,12:00:00,285,NaN,0.248953,NaN,0.338806,0.409216,NaN,...,0,0,lev20,1001,AAOT,45.3139,12.5083,10.0,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,13:10:2017,12:00:00,286,NaN,0.270292,NaN,0.344527,0.402378,NaN,...,0,0,lev20,1001,AAOT,45.3139,12.5083,10.0,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,14:10:2017,12:00:00,287,NaN,0.178536,NaN,0.242100,0.290174,NaN,...,0,0,lev20,1001,AAOT,45.3139,12.5083,10.0,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1419955,Zvenigorod,09:10:2021,12:00:00,282,NaN,0.032274,0.038148,NaN,NaN,0.045768,...,15,0,lev20,132,Zvenigorod,55.6950,36.7750,200.0,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419956,Zvenigorod,10:10:2021,12:00:00,283,NaN,0.019934,0.026304,NaN,NaN,0.034218,...,33,0,lev20,132,Zvenigorod,55.6950,36.7750,200.0,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419957,Zvenigorod,11:10:2021,12:00:00,284,NaN,0.029062,0.039846,NaN,NaN,0.057453,...,6,0,lev20,132,Zvenigorod,55.6950,36.7750,200.0,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419958,Zvenigorod,15:10:2021,12:00:00,288,NaN,0.027265,0.037933,NaN,NaN,0.057122,...,1,0,lev20,132,Zvenigorod,55.6950,36.7750,200.0,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov


In [35]:
aeronet_raw.columns

Index(['site', 'Date(dd:mm:yyyy)', 'Time(hh:mm:ss)', 'Day_of_Year',
       'AOD_1640nm', 'AOD_1020nm', 'AOD_870nm', 'AOD_865nm', 'AOD_779nm',
       'AOD_675nm', 'AOD_667nm', 'AOD_620nm', 'AOD_560nm', 'AOD_555nm',
       'AOD_551nm', 'AOD_532nm', 'AOD_531nm', 'AOD_510nm', 'AOD_500nm',
       'AOD_490nm', 'AOD_443nm', 'AOD_440nm', 'AOD_412nm', 'AOD_400nm',
       'AOD_380nm', 'AOD_340nm', 'Precipitable_Water(cm)', 'AOD_681nm',
       'AOD_709nm', 'AOD_Empty', 'AOD_Empty.1', 'AOD_Empty.2', 'AOD_Empty.3',
       'AOD_Empty.4', '440-870_Angstrom_Exponent', '380-500_Angstrom_Exponent',
       '440-675_Angstrom_Exponent', '500-870_Angstrom_Exponent',
       '340-440_Angstrom_Exponent', '440-675_Angstrom_Exponent[Polar]',
       'N[AOD_1640nm]', 'N[AOD_1020nm]', 'N[AOD_870nm]', 'N[AOD_865nm]',
       'N[AOD_779nm]', 'N[AOD_675nm]', 'N[AOD_667nm]', 'N[AOD_620nm]',
       'N[AOD_560nm]', 'N[AOD_555nm]', 'N[AOD_551nm]', 'N[AOD_532nm]',
       'N[AOD_531nm]', 'N[AOD_510nm]', 'N[AOD_500nm]', 'N[AO

## 3 · Tidy into an agent-ready table

* keep AERONET's own column names; only replace characters that break pandas / SQL / DuckDB (`Precipitable_Water(cm)` → `Precipitable_Water_cm`, `Site_Latitude(Degrees)` → `Site_Latitude_Degrees`, `N[AOD_500nm]` → `N_AOD_500nm`, `440-870_Angstrom_Exponent` → `Angstrom_Exponent_440_870nm`)
* add a proper `datetime_utc` (and `date`, `year`, `month`) built from AERONET's `dd:mm:yyyy` + `hh:mm:ss` columns
* drop the placeholder `AOD_Empty` columns and any wavelength that is missing at every site
* one consistent column order: site → time → location → AOD → Ångström → water vapour → observation counts → contact


In [36]:
def clean_col(name: str) -> str:
    name = re.sub(r"(\d{3})-(\d{3})_Angstrom_Exponent", r"Angstrom_Exponent_\1_\2nm", name)
    name = name.replace("[Polar]", "_Polar")
    name = re.sub(r"[\(\[]", "_", name)
    name = re.sub(r"[\)\]]", "", name)
    name = re.sub(r"[^0-9A-Za-z_]", "_", name)
    return re.sub(r"_+", "_", name).strip("_")

In [37]:
df = aeronet_raw.copy()

In [38]:
# ── placeholder columns (pandas mangles the duplicated "AOD_Empty" header to AOD_Empty, AOD_Empty.1, …)
df = df.drop(columns=[c for c in df.columns if "AOD_Empty" in c])

In [39]:
# AERONET stamps every daily average at 12:00:00 UTC
df["datetime_utc"] = pd.to_datetime(df["Date(dd:mm:yyyy)"] + " " + df["Time(hh:mm:ss)"], format="%d:%m:%Y %H:%M:%S")
df = df.drop(columns=["Date(dd:mm:yyyy)", "Time(hh:mm:ss)"])

df["date"]  = df["datetime_utc"].dt.normalize()
df["year"]  = df["datetime_utc"].dt.year.astype("int16")
df["month"] = df["datetime_utc"].dt.month.astype("int8")

In [40]:
df.columns = [clean_col(c) for c in df.columns]

In [41]:
# AERONET repeats the site name in every row – make sure it matches the file header, then keep the original column
mismatch = (df["AERONET_Site_Name"] != df["site"]).sum()
print(f"site-name mismatches between header and rows: {mismatch}")
df = df.drop(columns=["site"])


site-name mismatches between header and rows: 0


In [42]:
# ── drop wavelengths that no site ever measured (all-NaN) -----------------------
all_nan = [c for c in df.columns if df[c].isna().all()]
if all_nan:
    print(f"dropping {len(all_nan)} all-NaN columns: {all_nan}")
    df = df.drop(columns=all_nan)

dropping 1 all-NaN columns: ['AOD_531nm']


In [43]:
# … and their (now meaningless) observation-count columns
df = df.drop(columns=[c for c in df.columns if c.startswith("N_") and c[2:] not in df.columns])

In [44]:
# ── dtypes ----------------------------------------------------------------------
for c in df.columns:
    if c.startswith("N_"):
        df[c] = df[c].astype("Int32")
df["AERONET_Site_Name"] = df["AERONET_Site_Name"].astype("string")
for c in ("PI", "PI_Email", "Data_Quality_Level"):
    if c in df.columns:
        df[c] = df[c].astype("string")

In [45]:
# ── column order ----------------------------------------------------------------
def _group(c):
    order = ["AERONET_Site_Name", "datetime_utc", "date", "year", "month", "Day_of_Year",
             "Site_Latitude_Degrees", "Site_Longitude_Degrees", "Site_Elevation_m"]
    if c in order:                       return (0, order.index(c))
    if c.startswith("AOD_"):             return (1, -int(re.search(r"(\d+)nm", c).group(1)))   # long → short wavelength, like AERONET
    if c.startswith("Angstrom_"):        return (2, c)
    if c.startswith("Precipitable_"):    return (3, c)
    if c.startswith("N_"):               return (4, c)
    return (5, c)

In [46]:
df = df[sorted(df.columns, key=_group)]
df = df.sort_values(["AERONET_Site_Name", "datetime_utc"], ignore_index=True)

In [47]:
aeronet = df
del df
print(f"{aeronet.shape[0]:,} rows × {aeronet.shape[1]} columns")
print(list(aeronet.columns))

1,419,960 rows × 73 columns
['AERONET_Site_Name', 'datetime_utc', 'date', 'year', 'month', 'Day_of_Year', 'Site_Latitude_Degrees', 'Site_Longitude_Degrees', 'Site_Elevation_m', 'AOD_1640nm', 'AOD_1020nm', 'AOD_870nm', 'AOD_865nm', 'AOD_779nm', 'AOD_709nm', 'AOD_681nm', 'AOD_675nm', 'AOD_667nm', 'AOD_620nm', 'AOD_560nm', 'AOD_555nm', 'AOD_551nm', 'AOD_532nm', 'AOD_510nm', 'AOD_500nm', 'AOD_490nm', 'AOD_443nm', 'AOD_440nm', 'AOD_412nm', 'AOD_400nm', 'AOD_380nm', 'AOD_340nm', 'Angstrom_Exponent_340_440nm', 'Angstrom_Exponent_380_500nm', 'Angstrom_Exponent_440_675nm', 'Angstrom_Exponent_440_675nm_Polar', 'Angstrom_Exponent_440_870nm', 'Angstrom_Exponent_500_870nm', 'Precipitable_Water_cm', 'N_AOD_1020nm', 'N_AOD_1640nm', 'N_AOD_340nm', 'N_AOD_380nm', 'N_AOD_400nm', 'N_AOD_412nm', 'N_AOD_440nm', 'N_AOD_443nm', 'N_AOD_490nm', 'N_AOD_500nm', 'N_AOD_510nm', 'N_AOD_532nm', 'N_AOD_551nm', 'N_AOD_555nm', 'N_AOD_560nm', 'N_AOD_620nm', 'N_AOD_667nm', 'N_AOD_675nm', 'N_AOD_681nm', 'N_AOD_709nm', 'N_

In [48]:
aeronet

,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,AOD_1640nm,...,N_Angstrom_Exponent_380_500nm,N_Angstrom_Exponent_440_675nm,N_Angstrom_Exponent_440_675nm_Polar,N_Angstrom_Exponent_440_870nm,N_Angstrom_Exponent_500_870nm,N_Precipitable_Water_cm,AERONET_Instrument_Number,Data_Quality_Level,PI,PI_Email
0,AAOT,2017-10-04 12:00:00,2017-10-04,2017,10,277,45.3139,12.5083,10.0,NaN,...,0,25,0,25,0,25,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,2017-10-11 12:00:00,2017-10-11,2017,10,284,45.3139,12.5083,10.0,NaN,...,0,20,0,20,0,20,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,2017-10-12 12:00:00,2017-10-12,2017,10,285,45.3139,12.5083,10.0,NaN,...,0,33,0,33,0,33,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,2017-10-13 12:00:00,2017-10-13,2017,10,286,45.3139,12.5083,10.0,NaN,...,0,7,0,7,0,7,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,2017-10-14 12:00:00,2017-10-14,2017,10,287,45.3139,12.5083,10.0,NaN,...,0,12,0,12,0,12,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1419955,Zvenigorod,2021-10-09 12:00:00,2021-10-09,2021,10,282,55.6950,36.7750,200.0,NaN,...,15,15,0,15,15,15,132,lev20,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419956,Zvenigorod,2021-10-10 12:00:00,2021-10-10,2021,10,283,55.6950,36.7750,200.0,NaN,...,33,33,0,33,33,33,132,lev20,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419957,Zvenigorod,2021-10-11 12:00:00,2021-10-11,2021,10,284,55.6950,36.7750,200.0,NaN,...,6,6,0,6,6,6,132,lev20,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov
1419958,Zvenigorod,2021-10-15 12:00:00,2021-10-15,2021,10,288,55.6950,36.7750,200.0,NaN,...,1,1,0,1,1,1,132,lev20,Pawan_Gupta_and_Elena_Lind,pawan.gupta@nasa.gov_and_elena.lind@nasa.gov


## 4 · Sanity checks

In [49]:
print(f"sites        : {aeronet['AERONET_Site_Name'].nunique():,}")
print(f"rows         : {len(aeronet):,}")
print(f"time range   : {aeronet['datetime_utc'].min()}  →  {aeronet['datetime_utc'].max()}")
print(f"duplicates   : {aeronet.duplicated(['AERONET_Site_Name', 'datetime_utc']).sum():,}  (site × datetime)")
print(f"memory       : {aeronet.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

print("\nnon-null fraction per column:")
display((aeronet.notna().mean() * 100).round(1).rename("% non-null").to_frame().T)

display(aeronet.head())

sites        : 1,485
rows         : 1,419,960
time range   : 1993-05-14 12:00:00  →  2026-07-14 12:00:00
duplicates   : 0  (site × datetime)
memory       : 1,028 MB

non-null fraction per column:


,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,AOD_1640nm,...,N_Angstrom_Exponent_380_500nm,N_Angstrom_Exponent_440_675nm,N_Angstrom_Exponent_440_675nm_Polar,N_Angstrom_Exponent_440_870nm,N_Angstrom_Exponent_500_870nm,N_Precipitable_Water_cm,AERONET_Instrument_Number,Data_Quality_Level,PI,PI_Email
% non-null,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,53.5,...,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0


,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,AOD_1640nm,...,N_Angstrom_Exponent_380_500nm,N_Angstrom_Exponent_440_675nm,N_Angstrom_Exponent_440_675nm_Polar,N_Angstrom_Exponent_440_870nm,N_Angstrom_Exponent_500_870nm,N_Precipitable_Water_cm,AERONET_Instrument_Number,Data_Quality_Level,PI,PI_Email
0,AAOT,2017-10-04 12:00:00,2017-10-04,2017,10,277,45.3139,12.5083,10.0,NaN,...,0,25,0,25,0,25,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,2017-10-11 12:00:00,2017-10-11,2017,10,284,45.3139,12.5083,10.0,NaN,...,0,20,0,20,0,20,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,2017-10-12 12:00:00,2017-10-12,2017,10,285,45.3139,12.5083,10.0,NaN,...,0,33,0,33,0,33,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,2017-10-13 12:00:00,2017-10-13,2017,10,286,45.3139,12.5083,10.0,NaN,...,0,7,0,7,0,7,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,2017-10-14 12:00:00,2017-10-14,2017,10,287,45.3139,12.5083,10.0,NaN,...,0,12,0,12,0,12,1001,lev20,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu


## 5 · Export to `DATA/PROCESSED/`

One file: `AERONET_AOD_L2_Daily_V3.parquet` — the full merged table, one row per site × day.


In [50]:
aeronet.to_parquet(OUT_PARQUET, index=False, engine="pyarrow", compression="zstd")
print(f"wrote {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e6:,.0f} MB, {len(aeronet):,} rows × {aeronet.shape[1]} columns)")


wrote /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_AOD_L2_Daily_V3.parquet  (163 MB, 1,419,960 rows × 73 columns)


### Output schema (data dictionary for the agent's skill / system prompt)

Column names are AERONET's own, with only `( ) [ ] -` replaced by `_` so they work in pandas / SQL. Unit suffixes come from AERONET's headers.

| column | original AERONET header | unit | meaning |
|---|---|---|---|
| `AERONET_Site_Name` | `AERONET_Site_Name` | – | site name (string) |
| `datetime_utc`, `date`, `year`, `month` | *derived from* `Date(dd:mm:yyyy)` + `Time(hh:mm:ss)` | UTC | daily averages are stamped 12:00:00 UTC |
| `Day_of_Year` | `Day_of_Year` | day | 1–366 |
| `Site_Latitude_Degrees`, `Site_Longitude_Degrees` | `Site_Latitude(Degrees)`, `Site_Longitude(Degrees)` | decimal degrees | site location (WGS84) |
| `Site_Elevation_m` | `Site_Elevation(m)` | metres | site elevation above sea level |
| `AOD_<λ>nm` | `AOD_<λ>nm` | unitless | aerosol optical depth at wavelength λ nanometres; NaN = not measured / did not pass Level 2.0 screening |
| `Angstrom_Exponent_<λ1>_<λ2>nm` | `<λ1>-<λ2>_Angstrom_Exponent` | unitless | Ångström exponent computed between λ1 and λ2 nm (`_Polar` = polar-instrument variant) |
| `Precipitable_Water_cm` | `Precipitable_Water(cm)` | centimetres | total column water vapour |
| `N_<column>` | `N[<column>]` | count | number of Level 2.0 measurements averaged into that daily value (0 ⇒ value is NaN) |
| `Data_Quality_Level` | `Data_Quality_Level` | – | always `lev20` |
| `AERONET_Instrument_Number` | `AERONET_Instrument_Number` | – | CIMEL sun-photometer id |
| `PI`, `PI_Email` | *file header line 5* | – | site principal investigator (AERONET data-use policy: contact / acknowledge) |

Units reference: <https://aeronet.gsfc.nasa.gov/new_web/units.html>
